In [2]:
import lightgbm as lgb
import xgboost as xgb
import optuna
import numpy as np


c:\Users\84983\Downloads\aio_conquer_3\hybrid-forecasting-gb-pretrained\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
#Import libraries, and self-defined functions from models_gb.py, features.py, data_loader.py
import sys
import importlib
sys.path.append('../src')
import models_gb
importlib.reload(models_gb)
from models_gb import (
    train_lightgbm, train_xgboost,
    predict_lightgbm, predict_xgboost,
    calculate_metrics, get_feature_importance_lgb
)
from features import create_time_features_optimized, make_train_test_split
import data_loader
importlib.reload(data_loader)
from data_loader import load_and_preprocess_m4_monthly
import pandas as pd

#Load a manageable subset for the baseline experiment.
#Set max_series=None only when enough RAM is available for the full dataset.
train_df, test_df = load_and_preprocess_m4_monthly(
    "../data/raw/Train/Monthly-train.csv",
    "../data/raw/Test/Monthly-test.csv",
    max_series=500
)

#Create Features
train_df = create_time_features_optimized(train_df)
test_df = create_time_features_optimized(test_df)

#Split Train/Val
train_split, val_split = make_train_test_split(train_df, test_horizon=18)

#Prepare Features
feature_cols = [col for col in train_split.columns
                if col not in ['unique_id', 'ds', 'y']]
X_train = train_split[feature_cols].bfill()
y_train = train_split['y']
X_val = val_split[feature_cols].bfill()
y_val = val_split['y']

#Train LightGBM
lgb_model = train_lightgbm(X_train, y_train, X_val, y_val)

#Train XGBoost
xgb_model = train_xgboost(X_train, y_train, X_val, y_val)

#Evaluate
lgb_pred = predict_lightgbm(lgb_model, X_val)
lgb_metrics = calculate_metrics(y_val, lgb_pred)
print("LightGBM Metrics:", lgb_metrics)

#Feature Importance
importance = get_feature_importance_lgb(lgb_model, top_n=15)
print(importance)

--- Bước 1 & 2: Đọc và Melt dữ liệu ---
--- Bước 3: Tái tạo mốc thời gian ---
Hoàn thành! Kích thước Train: (158843, 3), Test: (9000, 3)
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 778.938
Early stopping, best iteration is:
[128]	valid_0's rmse: 774.887
[0]	eval-rmse:4958.76266
[100]	eval-rmse:769.26596
[200]	eval-rmse:763.39226
[251]	eval-rmse:764.42716
LightGBM Metrics: {'MAE': 330.4493678444652, 'RMSE': np.float64(774.8869729880308), 'MAPE': np.float64(6.555248248966286)}
            feature  importance
4             lag_1         859
8            lag_12         492
13  rolling_mean_12         281
10    rolling_std_3         275
6             lag_3         268
9    rolling_mean_3         247
5             lag_2         239
0              year         221
1             month         214
7             lag_6         207
14   rolling_std_12         202
11   rolling_mean_6         168
12    rolling_std_6         136
2           quarter          14
3